# 03 — Evaluation

This notebook documents Phase 3: evaluating the fine-tuned ChatTLA model and comparing it to the base `gpt-oss:20b` baseline.

## Evaluation Dimensions

| Metric | Description | Tool |
|--------|-------------|------|
| `sany_parse_rate` | % generated specs that are syntactically valid TLA+ | SANY (tla2tools.jar) |
| `tlc_clean_rate` | % generated specs that TLC model-checks with no violations | TLC (tla2tools.jar) |
| `structural_score` | Heuristic rubric: has MODULE, EXTENDS, VARIABLES, Init, Next, Spec, TypeOK, expected invariants | `src/inference/benchmark.py` |

## Benchmark Suite

20 hand-crafted problems in `data/benchmarks/benchmark_suite.json`, covering:
- Consensus (Paxos, Raft, Bakery)
- Transactions (2PC, Snapshot Isolation)
- Scheduling (Dining Philosophers, Peterson's, Dekker's)
- Storage (KV Store, Allocator, G-Counter CRDT)
- Networking (Chandy-Lamport, Gossip, Token Ring)

Difficulty range: 2 (beginner) — 5 (research-grade).

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

REPO_ROOT = Path('.').resolve()

with (REPO_ROOT / 'data' / 'benchmarks' / 'benchmark_suite.json').open() as f:
    benchmarks = json.load(f)

bdf = pd.DataFrame(benchmarks)
print(f'{len(bdf)} benchmark problems')
bdf[['id','name','domain','difficulty']].head(20)

## Step 1: Verify Ollama is Running

The fine-tuned model must be registered first:
```bash
ollama pull gpt-oss:20b           # base baseline
python -m src.inference.convert_to_gguf   # build chattla:20b from merged model
ollama list                        # verify both models are present
```

In [ ]:
import subprocess
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print(result.stdout)
if 'gpt-oss:20b' not in result.stdout:
    print('WARNING: gpt-oss:20b not found. Run: ollama pull gpt-oss:20b')
if 'chattla:20b' not in result.stdout:
    print('WARNING: chattla:20b not found. Run: python -m src.inference.convert_to_gguf')

## Step 2: Run Benchmark Suite

Compare base `gpt-oss:20b` vs fine-tuned `chattla:20b` on all 20 problems.

This takes ~30-60 min depending on hardware. Results are written to `outputs/benchmark_results.csv`.

Alternatively run from terminal:
```bash
python -m src.inference.benchmark                        # both models
python -m src.inference.benchmark --model chattla:20b    # fine-tuned only
python -m src.inference.benchmark --self-correct         # with TLC feedback loop
```

In [ ]:
# Uncomment to run benchmark from notebook:
# from src.inference.benchmark import run
# run()

## Step 3: Analyse Results

In [ ]:
results_csv = REPO_ROOT / 'outputs' / 'benchmark_results.csv'
if not results_csv.exists():
    print('benchmark_results.csv not yet generated — run the benchmark first.')
else:
    df = pd.read_csv(results_csv)
    print(f'Models evaluated: {df.model.unique()}')
    print(f'Problems: {df.benchmark_id.nunique()}')

    summary = df.groupby('model').agg(
        sany_pass_rate    =('sany_pass',         'mean'),
        tlc_pass_rate     =('tlc_pass',          'mean'),
        avg_structural    =('structural_score',  'mean'),
        avg_runtime_s     =('runtime_s',         'mean'),
    ).round(3)
    print('\n=== Summary ===')
    print(summary.to_string())

    # Per-difficulty breakdown
    diff_df = df.merge(bdf[['id','difficulty']], left_on='benchmark_id', right_on='id')
    diff_summary = diff_df.groupby(['model','difficulty'])['tlc_pass'].mean().unstack()
    print('\n=== TLC pass rate by difficulty ===')
    print(diff_summary.round(2).to_string())

In [ ]:
# Visualise results
if results_csv.exists():
    models = df['model'].unique()
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    metrics = ['sany_pass', 'tlc_pass', 'structural_score']
    titles  = ['SANY Parse Rate', 'TLC Clean Rate', 'Structural Score']

    for ax, metric, title in zip(axes, metrics, titles):
        domain_df = df.groupby(['model', 'domain'])[metric].mean().unstack()
        domain_df.T.plot(kind='bar', ax=ax, title=title)
        ax.set_ylabel('Score')
        ax.set_xlabel('Domain')
        ax.tick_params(axis='x', rotation=30)
        if 'rate' in metric:
            ax.set_ylim(0, 1)
            ax.axhline(0.70, ls='--', color='red', lw=0.8, label='70% target' if 'tlc' in metric else '')

    plt.tight_layout()
    plt.savefig(REPO_ROOT / 'outputs' / 'benchmark_by_domain.png', dpi=150)
    plt.show()
    print('Plot saved to outputs/benchmark_by_domain.png')

## Step 4: Qualitative Analysis

Examining generated specs for specific benchmark problems helps identify failure modes.
Common issues to look for:
- Missing `Spec == Init /\ [][Next]_vars`
- Wrong module name (must match filename for SANY)
- Hallucinated TLA+ operators (e.g. `FORALL` instead of `\\A`)
- State space explosion (TLC times out → silver not gold)

In [ ]:
# Inspect a specific generated spec
if results_csv.exists():
    # Show the first TLC-failed ChatTLA result for analysis
    failed = df[(df['model'] == 'chattla:20b') & (df['tlc_pass'] == 0)]
    if not failed.empty:
        row = failed.iloc[0]
        print(f"Problem: {row['name']} (difficulty={row['difficulty']})")
        print(f"TLC tier: {row['tlc_tier']}")
        print(f"Structural score: {row['structural_score']}")
        print('\n--- Generated Spec ---')
        print(row['generated_spec'])
    else:
        print('All ChatTLA problems passed TLC! ')